In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.ensemble import RandomForestClassifier

import joblib


In [ ]:

train = pd.read_csv("/home/sanjay/Internship/datasets/fraudTrain.csv",encoding='unicode_escape')
test = pd.read_csv("/home/sanjay/Internship/datasets/fraudTest.csv",encoding='unicode_escape')

print("Train Shape:", train.shape)
print("Test Shape:", test.shape)
train.head()


Train Shape: (1296675, 23)
Test Shape: (555719, 23)


,Unnamed: 0,trans_date_trans_time,cc_num,merchant,category,amt,first,last,gender,street,...,lat,long,city_pop,job,dob,trans_num,unix_time,merch_lat,merch_long,is_fraud
0,0,2019-01-01 00:00:18,2703186189652095,"fraud_Rippin, Kub and Mann",misc_net,4.97,Jennifer,Banks,F,561 Perry Cove,...,36.0788,-81.1781,3495,"Psychologist, counselling",1988-03-09,0b242abb623afc578575680df30655b9,1325376018,36.011293,-82.048315,0
1,1,2019-01-01 00:00:44,630423337322,"fraud_Heller, Gutmann and Zieme",grocery_pos,107.23,Stephanie,Gill,F,43039 Riley Greens Suite 393,...,48.8878,-118.2105,149,Special educational needs teacher,1978-06-21,1f76529f8574734946361c461b024d99,1325376044,49.159047,-118.186462,0
2,2,2019-01-01 00:00:51,38859492057661,fraud_Lind-Buckridge,entertainment,220.11,Edward,Sanchez,M,594 White Dale Suite 530,...,42.1808,-112.2620,4154,Nature conservation officer,1962-01-19,a1a22d70485983eac12b5b88dad1cf95,1325376051,43.150704,-112.154481,0
3,3,2019-01-01 00:01:16,3534093764340240,"fraud_Kutch, Hermiston and Farrell",gas_transport,45.00,Jeremy,White,M,9443 Cynthia Court Apt. 038,...,46.2306,-112.1138,1939,Patent attorney,1967-01-12,6b849c168bdad6f867558c3793159a81,1325376076,47.034331,-112.561071,0
4,4,2019-01-01 00:03:06,375534208663984,fraud_Keeling-Crist,misc_pos,41.96,Tyler,Garcia,M,408 Bradley Rest,...,38.4207,-79.4629,99,Dance movement psychotherapist,1986-03-28,a41d7549acf90789359a9aa5346dcb46,1325376186,38.674999,-78.632459,0


In [ ]:
print(train.info()) 




<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1296675 entries, 0 to 1296674
Data columns (total 23 columns):
 #   Column                 Non-Null Count    Dtype  
---  ------                 --------------    -----  
 0   Unnamed: 0             1296675 non-null  int64  
 1   trans_date_trans_time  1296675 non-null  object 
 2   cc_num                 1296675 non-null  int64  
 3   merchant               1296675 non-null  object 
 4   category               1296675 non-null  object 
 5   amt                    1296675 non-null  float64
 6   first                  1296675 non-null  object 
 7   last                   1296675 non-null  object 
 8   gender                 1296675 non-null  object 
 9   street                 1296675 non-null  object 
 10  city                   1296675 non-null  object 
 11  state                  1296675 non-null  object 
 12  zip                    1296675 non-null  int64  
 13  lat                    1296675 non-null  float64
 14  long              

In [7]:
train.isna().sum()

Unnamed: 0               0
trans_date_trans_time    0
cc_num                   0
merchant                 0
category                 0
amt                      0
first                    0
last                     0
gender                   0
street                   0
city                     0
state                    0
zip                      0
lat                      0
long                     0
city_pop                 0
job                      0
dob                      0
trans_num                0
unix_time                0
merch_lat                0
merch_long               0
is_fraud                 0
dtype: int64

In [4]:
train.describe()

,Unnamed: 0,cc_num,amt,zip,lat,long,city_pop,unix_time,merch_lat,merch_long,is_fraud
count,1.296675e+06,1.296675e+06,1.296675e+06,1.296675e+06,1.296675e+06,1.296675e+06,1.296675e+06,1.296675e+06,1.296675e+06,1.296675e+06,1.296675e+06
mean,6.483370e+05,4.171920e+17,7.035104e+01,4.880067e+04,3.853762e+01,-9.022634e+01,8.882444e+04,1.349244e+09,3.853734e+01,-9.022646e+01,5.788652e-03
std,3.743180e+05,1.308806e+18,1.603160e+02,2.689322e+04,5.075808e+00,1.375908e+01,3.019564e+05,1.284128e+07,5.109788e+00,1.377109e+01,7.586269e-02
min,0.000000e+00,6.041621e+10,1.000000e+00,1.257000e+03,2.002710e+01,-1.656723e+02,2.300000e+01,1.325376e+09,1.902779e+01,-1.666712e+02,0.000000e+00
25%,3.241685e+05,1.800429e+14,9.650000e+00,2.623700e+04,3.462050e+01,-9.679800e+01,7.430000e+02,1.338751e+09,3.473357e+01,-9.689728e+01,0.000000e+00
50%,6.483370e+05,3.521417e+15,4.752000e+01,4.817400e+04,3.935430e+01,-8.747690e+01,2.456000e+03,1.349250e+09,3.936568e+01,-8.743839e+01,0.000000e+00
75%,9.725055e+05,4.642255e+15,8.314000e+01,7.204200e+04,4.194040e+01,-8.015800e+01,2.032800e+04,1.359385e+09,4.195716e+01,-8.023680e+01,0.000000e+00
max,1.296674e+06,4.992346e+18,2.894890e+04,9.978300e+04,6.669330e+01,-6.795030e+01,2.906700e+06,1.371817e+09,6.751027e+01,-6.695090e+01,1.000000e+00


In [5]:
train['is_fraud'].value_counts()

is_fraud
0    1289169
1       7506
Name: count, dtype: int64

In [ ]:
 
if "trans_date_trans_time" in train.columns:
    train["trans_date_trans_time"] = pd.to_datetime(train["trans_date_trans_time"])
    test["trans_date_trans_time"] = pd.to_datetime(test["trans_date_trans_time"])

    for df in [train, test]:
        df["hour"] = df["trans_date_trans_time"].dt.hour
        df["day"] = df["trans_date_trans_time"].dt.day
        df["month"] = df["trans_date_trans_time"].dt.month
        df["year"] = df["trans_date_trans_time"].dt.year

    train = train.drop(columns=["trans_date_trans_time"])
    test = test.drop(columns=["trans_date_trans_time"])

✅ Preprocessing Completed Successfully (No Unknown Label Errors)


In [ ]:
X = train.drop("is_fraud", axis=1)
y = train["is_fraud"]


In [ ]:
num_cols = X.select_dtypes(include=["int64", "float64"]).columns
cat_cols = X.select_dtypes(include=["object"]).columns


In [ ]:
for col in num_cols:
    X[col].fillna(X[col].median(), inplace=True)
    test[col].fillna(test[col].median(), inplace=True)

for col in cat_cols:
    X[col].fillna("Unknown", inplace=True)
    test[col].fillna("Unknown", inplace=True)


for col in cat_cols:
    le = LabelEncoder()
    combined = pd.concat([X[col], test[col]], axis=0).astype(str)
    le.fit(combined)

    X[col] = le.transform(X[col].astype(str))
    test[col] = le.transform(test[col].astype(str))


In [ ]:
scaler = StandardScaler()
X[num_cols] = scaler.fit_transform(X[num_cols])
test[num_cols] = scaler.transform(test[num_cols])

print("Preprocessing Completed Successfully (No Unknown Label Errors)")
 


In [30]:
X_train,X_val,y_train,y_val=train_test_split(X,y,random_state=42,test_size=0.25) 

In [31]:
model = RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced')


In [32]:
model.fit(X_train, y_train)   

RandomForestClassifier(class_weight='balanced', n_estimators=200,
                       random_state=42)

In [ ]:

y_pred = model.predict(X_val)

print("\nClassification Report:\n")
print(classification_report(y_val, y_pred))

print("\nConfusion Matrix:\n")
print(confusion_matrix(y_val, y_pred))

roc = roc_auc_score(y_val, model.predict_proba(X_val)[:,1])
print("\nROC-AUC Score:", roc)
 


Classification Report:

              precision    recall  f1-score   support

           0       1.00      1.00      1.00    322285
           1       0.98      0.71      0.83      1884

    accuracy                           1.00    324169
   macro avg       0.99      0.86      0.91    324169
weighted avg       1.00      1.00      1.00    324169


Confusion Matrix:

[[322262     23]
 [   542   1342]]

ROC-AUC Score: 0.9972756858890471


In [ ]:

test['fraud_probability'] = model.predict_proba(test)[:,1]
test['is_fraud_predicted'] = model.predict(test)

test[['fraud_probability', 'is_fraud_predicted']].head()


test.to_csv("fraud_predictions.csv", index=False)
print("Predictions saved as fraud_predictions.csv")
   

ValueError: The feature names should match those that were passed during fit.
Feature names unseen at fit time:
- is_fraud
